In [ ]:
# Colab setup - run this first. Installs m3 plus the tutorial plotting extras.
# (Colab already ships PyTorch, which m3 uses as its engine.)
%pip install -q "git+https://github.com/PYangLab/M3.git" scanpy umap-learn

# Feature attribution

IG attribution on the built-in Liu subsample. Top genes/celltypes you'll see here
are consistent in *pattern* with the full-data publication ranking (Mono + NK
in top, T-cell IFN/cytotoxic markers in top genes) but specific rank order may
differ because of the smaller sample. All hyperparameters are identical to the
full-data run (max_epochs=500, n_steps=50, min_cells_per_condition=200); only
the data is subsampled.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import m3

OUT = "./tutorial_out_demo/03_attribution"
os.makedirs(OUT, exist_ok=True)

In [ ]:
data = m3.datasets.liu_demo()
print(data)

In [ ]:
model = m3.M3(
    data,
    condition_keys=["cond_group", "Age_interval"],
    target_condition="cond_group",
    celltype_key="mergedcelltype",
    batch_key="batch",
    donor_key="sample_id",
    embedding_dim=30,
)
#To save time, users can set max_epochs to 100 for test.
model.train(max_epochs=500)
print("capabilities:", model.capabilities)

## Attribute: Severe vs HC

In [ ]:
attr = model.attribute(reference_labels=["HC"])
print(attr)
attr.celltypes.to_csv(os.path.join(OUT, "celltype_importance.csv"), index=False)
if attr.donors is not None:
    attr.donors.to_csv(os.path.join(OUT, "donor_importance.csv"), index=False)
print("top cell types (filtered: >= 200 cells per condition):\n",
      attr.top_celltypes(min_cells_per_condition=200).to_string(index=False))

## Publication-recipe top genes
`top_genes(min_cells_per_condition=200, modality='rna')` filters by cell-type
balance and excludes housekeeping / ribosomal genes by name — identical settings
to the full-data run.

In [ ]:
top_rna = attr.top_genes(n=100, min_cells_per_condition=200, modality="rna")
print(f"top-100 RNA genes (computed over {int(top_rna['n_celltypes_used'].iloc[0])} balanced cell types):")
print(top_rna.head(15).to_string(index=False))
top_rna.to_csv(os.path.join(OUT, "gene_importance_top100_rna.csv"), index=False)

## Save attribution matrix

In [ ]:
import h5py
import anndata as ad

attr_mat = attr.attribution
present = [m for m in ("rna", "adt", "atac") if m in data.modality_names]
feat_names = [f for mm in present for f in list(data.var[mm])][: attr_mat.shape[1]]

with h5py.File(os.path.join(OUT, "attribution.h5"), "w") as f:
    f["data"] = attr_mat

_obs = model.cell_metadata.astype(str).reset_index(drop=True)
ad.AnnData(X=attr_mat, obs=_obs, var=pd.DataFrame(index=feat_names)).write_h5ad(
    os.path.join(OUT, "attribution.h5ad"))
print("saved attribution -> .h5 (['data']) / .h5ad in", OUT)

## Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
g = top_rna.head(20).iloc[::-1]
axes[0].barh(g["feature"], g["score"], color="#4c72b0")
axes[0].set_title("Top-20 RNA genes (per-celltype-balanced)")
axes[0].tick_params(axis="y", labelsize=7)
axes[0].set_xlabel("mean |IG| across balanced cell types")

c = attr.top_celltypes(min_cells_per_condition=200).iloc[::-1]
axes[1].barh(c["celltype"], c["importance"], color="#dd8452")
axes[1].set_title("Cell-type importance")
axes[1].tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "importance_bars.png"), dpi=130, bbox_inches="tight")
plt.show()

# Gene x celltype heatmap (top genes)
gcm = attr.gene_celltype_matrix
top_features = top_rna.head(30)["feature"].tolist()
feat_idx = {n: i for i, n in enumerate(attr._feature_names)}
top_gene_pos = [feat_idx[f] for f in top_features]
sub = gcm[:, top_gene_pos]
ct_names = list(attr._res["celltype_names"])

fig2, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(sub, aspect="auto", cmap="RdBu_r",
               vmin=-np.nanmax(np.abs(sub)), vmax=np.nanmax(np.abs(sub)))
ax.set_xticks(range(len(top_gene_pos)))
ax.set_xticklabels(top_features, rotation=90, fontsize=6)
ax.set_yticks(range(len(ct_names)))
ax.set_yticklabels(ct_names, fontsize=7)
ax.set_title("Gene × cell-type attribution (top-30 RNA genes)")
fig2.colorbar(im, ax=ax, fraction=0.02, label="signed IG")
fig2.tight_layout()
fig2.savefig(os.path.join(OUT, "gene_celltype_heatmap.png"), dpi=130, bbox_inches="tight")
plt.show()